# 9.2 — Quantos y composites

Dos formas distintas de exponerse a un activo extranjero desde la moneda doméstica:
**quanto** (paga en doméstica un monto fijado a un tipo de cambio PREACORDADO, sin riesgo
FX) y **composite** (paga en doméstica el valor CONVERTIDO al FX realizado, con riesgo FX
pleno). Ambos se reducen a Black-Scholes tras el ajuste correcto — pero el ajuste de
drift del quanto (Girsanov, 1.5) es fácil de aplicar mal, como se documenta abajo.

## 0. Configuración

Activo extranjero $S_t$ (en moneda extranjera), FX $X_t$ (unidades domésticas por
extranjera). $r_f$ = tasa libre de riesgo extranjera, $q$ = dividend yield del activo
extranjero, $r_d$ = tasa doméstica.

In [1]:
import numpy as np
from scipy.stats import norm

rng = np.random.default_rng(42)

S0, X0 = 100.0, 1.0
r_f, q, r_d = 0.03, 0.01, 0.05
sigma_S, sigma_FX = 0.25, 0.12
T, K = 1.0, 100.0

## 1. Quanto: ajuste de drift, misma volatilidad

Bajo la medida risk-neutral EXTRANJERA, $dS_t=S_t\left((r_f-q)dt+\sigma_S dW_t^S\right)$.
Un quanto paga en doméstica al tipo de cambio FIJO $\bar X$: el cambio de medida
(Girsanov, de la medida extranjera a la doméstica) introduce un ajuste de drift por la
covarianza activo-FX, **sin cambiar la volatilidad**:
$$dS_t = S_t\left((r_f-q-\rho\,\sigma_S\sigma_{FX})dt+\sigma_S dW_t^S\right) \quad \text{(bajo la medida domestica)}$$
El precio de una call quanto es entonces Black-Scholes con esa tasa efectiva
$r_f-q-\rho\sigma_S\sigma_{FX}$ en vez de $r_f-q$, descontado a $r_d$ (no a $r_f$).

In [2]:
def quanto_call_price(S0, K, r_d, drift_adj, sigma_S, T, X_fixed=1.0):
    d1 = (np.log(S0 / K) + (drift_adj + 0.5 * sigma_S ** 2) * T) / (sigma_S * np.sqrt(T))
    d2 = d1 - sigma_S * np.sqrt(T)
    return X_fixed * np.exp(-r_d * T) * (S0 * np.exp(drift_adj * T) * norm.cdf(d1) - K * norm.cdf(d2))


def simulate_quanto_mc(rho, drift_adj, n_paths, n_steps, rng):
    dt = T / n_steps
    chol = np.array([[1.0, 0.0], [rho, np.sqrt(1 - rho ** 2)]])
    S = np.full(n_paths, S0)
    for _ in range(n_steps):
        Z = rng.standard_normal((n_paths, 2))
        dW = (Z @ chol.T) * np.sqrt(dt)
        S = S * np.exp((drift_adj - 0.5 * sigma_S ** 2) * dt + sigma_S * dW[:, 0])
    return S

### Demo 1 — quanto: fórmula cerrada vs MC bivariado, para $\rho>0$ y $\rho<0$

In [3]:
n_paths, n_steps = 300_000, 100
results_quanto = {}
for rho in [-0.5, 0.0, 0.5]:
    drift_adj = r_f - q - rho * sigma_S * sigma_FX
    price_closed = quanto_call_price(S0, K, r_d, drift_adj, sigma_S, T)
    S_T = simulate_quanto_mc(rho, drift_adj, n_paths, n_steps, rng)
    payoff = np.maximum(S_T - K, 0.0) * np.exp(-r_d * T)
    price_mc, se_mc = payoff.mean(), payoff.std() / np.sqrt(n_paths)
    z = (price_mc - price_closed) / se_mc
    results_quanto[rho] = (price_closed, price_mc, se_mc, z)
    print(f"rho={rho:+.1f}: cerrado={price_closed:.4f}  MC={price_mc:.4f}+-{se_mc:.4f}  z={z:.2f}")

print("\nel ajuste de drift crece en magnitud con |rho| (y cambia de signo con rho) --")
print("el 'quanto skew' que en la practica exige cotizar quantos con su propia correlacion implicita.")

rho=-0.5: cerrado=11.4189  MC=11.4588+-0.0325  z=1.23


rho=+0.0: cerrado=10.5493  MC=10.6253+-0.0313  z=2.43


rho=+0.5: cerrado=9.7265  MC=9.7097+-0.0299  z=-0.56

el ajuste de drift crece en magnitud con |rho| (y cambia de signo con rho) --
el 'quanto skew' que en la practica exige cotizar quantos con su propia correlacion implicita.


### Demo 2 — límite $\rho=0$: sin ajuste de drift

In [4]:
drift_adj_rho0 = r_f - q - 0.0 * sigma_S * sigma_FX
print(f"drift ajustado con rho=0: {drift_adj_rho0:.4f}  (== r_f-q sin ajuste: {r_f - q:.4f})")
assert abs(drift_adj_rho0 - (r_f - q)) < 1e-12

drift ajustado con rho=0: 0.0200  (== r_f-q sin ajuste: 0.0200)


## 2. Composite: volatilidad combinada, sin ajuste de drift adicional

Un composite paga en doméstica el valor REALIZADO $S_T X_T$ (activo extranjero convertido
al FX del momento) — el subyacente relevante es el producto $S_tX_t$. Por Itô,
$d(SX)=SX\left[(\mu_S+\mu_X+\rho\sigma_S\sigma_{FX})dt+\sigma_S dW^S+\sigma_{FX}dW^X\right]$;
bajo la medida doméstica, $S_t$ lleva el MISMO ajuste de drift que en el quanto (es la
misma medida), y $\mu_X=r_d-r_f$ (GK estándar) — sumando todo, el drift de $S_tX_t$ bajo
la medida doméstica colapsa exactamente a $r_d-q$ (sin residuo de $\rho$: se cancela), y
la volatilidad total es
$$\sigma_{comp}=\sqrt{\sigma_S^2+\sigma_{FX}^2+2\rho\,\sigma_S\sigma_{FX}}$$
— de nuevo Black-Scholes, ahora sobre $F_0=S_0X_0$ con esta vol combinada.

**Nota de implementación (bug real encontrado al prototipar):** en la simulación MC de
control, $S_t$ debe simularse bajo la medida DOMÉSTICA (con el MISMO ajuste de drift
$-\rho\sigma_S\sigma_{FX}$ del quanto) — usar el drift extranjero puro $r_f-q$ para $S_t$
(el error natural, ya que $S_t$ "vive" en el mundo extranjero) da una discrepancia MC vs
fórmula cerrada de $z\approx\pm25$ para $\rho\ne0$ (coincide sólo en $\rho=0$, donde el
ajuste es cero) — la firma clásica de un olvido de Girsanov al cambiar de medida.

In [5]:
def composite_call_price(S0, X0, K, r_d, q, sigma_comp, T):
    F0 = S0 * X0
    d1 = (np.log(F0 / K) + (r_d - q + 0.5 * sigma_comp ** 2) * T) / (sigma_comp * np.sqrt(T))
    d2 = d1 - sigma_comp * np.sqrt(T)
    return F0 * np.exp(-q * T) * norm.cdf(d1) - K * np.exp(-r_d * T) * norm.cdf(d2)


def simulate_composite_mc(rho, n_paths, n_steps, rng):
    dt = T / n_steps
    chol = np.array([[1.0, 0.0], [rho, np.sqrt(1 - rho ** 2)]])
    S = np.full(n_paths, S0)
    X = np.full(n_paths, X0)
    drift_S_domestic = r_f - q - rho * sigma_S * sigma_FX  # -- el ajuste de Girsanov, NO r_f-q puro
    for _ in range(n_steps):
        Z = rng.standard_normal((n_paths, 2))
        dW = (Z @ chol.T) * np.sqrt(dt)
        S = S * np.exp((drift_S_domestic - 0.5 * sigma_S ** 2) * dt + sigma_S * dW[:, 0])
        X = X * np.exp((r_d - r_f - 0.5 * sigma_FX ** 2) * dt + sigma_FX * dW[:, 1])
    return S, X

### Demo 3 — composite: fórmula cerrada vs MC del producto $S_TX_T$

In [6]:
results_composite = {}
for rho in [-0.5, 0.0, 0.5]:
    sigma_comp = np.sqrt(sigma_S ** 2 + sigma_FX ** 2 + 2 * rho * sigma_S * sigma_FX)
    price_closed = composite_call_price(S0, X0, K, r_d, q, sigma_comp, T)
    S_T, X_T = simulate_composite_mc(rho, n_paths, n_steps, rng)
    payoff = np.maximum(S_T * X_T - K, 0.0) * np.exp(-r_d * T)
    price_mc, se_mc = payoff.mean(), payoff.std() / np.sqrt(n_paths)
    z = (price_mc - price_closed) / se_mc
    results_composite[rho] = (price_closed, price_mc, se_mc, z)
    print(f"rho={rho:+.1f}: sigma_comp={sigma_comp:.4f}  cerrado={price_closed:.4f}  "
          f"MC={price_mc:.4f}+-{se_mc:.4f}  z={z:.2f}")

rho=-0.5: sigma_comp=0.2166  cerrado=10.4524  MC=10.4432+-0.0283  z=-0.33


rho=+0.0: sigma_comp=0.2773  cerrado=12.7553  MC=12.7230+-0.0369  z=-0.88


rho=+0.5: sigma_comp=0.3270  cerrado=14.6389  MC=14.7016+-0.0444  z=1.41


### Demo 4 — límite $\rho=0$: vol combinada colapsa a $\sqrt{\sigma_S^2+\sigma_{FX}^2}$

In [7]:
sigma_comp_rho0 = np.sqrt(sigma_S ** 2 + sigma_FX ** 2 + 2 * 0.0 * sigma_S * sigma_FX)
sigma_comp_expected = np.sqrt(sigma_S ** 2 + sigma_FX ** 2)
diff_comp_limit = abs(sigma_comp_rho0 - sigma_comp_expected)
print(f"sigma_comp(rho=0)={sigma_comp_rho0:.10f}  vs  sqrt(sigma_S^2+sigma_FX^2)={sigma_comp_expected:.10f}  "
      f"diff={diff_comp_limit:.2e}")

sigma_comp(rho=0)=0.2773084925  vs  sqrt(sigma_S^2+sigma_FX^2)=0.2773084925  diff=0.00e+00


## 3. Validación

In [8]:
zs_quanto = np.array([results_quanto[rho][3] for rho in [-0.5, 0.0, 0.5]])
check1 = np.all(np.abs(zs_quanto) <= 3.0)
print(f"1. quanto: cerrado vs MC dentro de 3 SE (rho=-0.5,0,0.5): {check1} (max|z|={np.max(np.abs(zs_quanto)):.2f})")

check2 = abs(drift_adj_rho0 - (r_f - q)) < 1e-12
print(f"2. limite rho=0 (quanto): sin ajuste de drift, atol=1e-12: {check2}")

zs_comp = np.array([results_composite[rho][3] for rho in [-0.5, 0.0, 0.5]])
check3 = np.all(np.abs(zs_comp) <= 3.0)
print(f"3. composite: cerrado vs MC dentro de 3 SE (rho=-0.5,0,0.5): {check3} (max|z|={np.max(np.abs(zs_comp)):.2f})")

check4 = diff_comp_limit < 1e-10
print(f"4. limite rho=0 (composite): sigma_comp=sqrt(sigma_S^2+sigma_FX^2), atol=1e-10: {check4}")

assert check1
assert check2
assert check3
assert check4
print("\nValidacion completa: 4/4 dentro de tolerancia.")

1. quanto: cerrado vs MC dentro de 3 SE (rho=-0.5,0,0.5): True (max|z|=2.43)
2. limite rho=0 (quanto): sin ajuste de drift, atol=1e-12: True
3. composite: cerrado vs MC dentro de 3 SE (rho=-0.5,0,0.5): True (max|z|=1.41)
4. limite rho=0 (composite): sigma_comp=sqrt(sigma_S^2+sigma_FX^2), atol=1e-10: True

Validacion completa: 4/4 dentro de tolerancia.


## Referencias

- Reiner, E. (1992). *Quanto Mechanics*. Risk, 5(3), 59-63.
- Wystup, U. (2006). *FX Options and Structured Products*, cap. 4.
- Hull, J. *Options, Futures, and Other Derivatives*, sección sobre opciones quanto.